# Harmful Brain Activity Classification (HMS Kaggle Dataset)

This notebook is our preliminary workflow for the DATA 3421 final project.  
Our goal is to classify harmful brain activity patterns from EEG data by:

- exploring the metadata,
- extracting labeled 10-second EEG segments,
- engineering tabular signal features,
- comparing baseline and stronger machine learning models.

We keep the workflow simple and readable so we can turn it into our final report and presentation later.

In [2]:

import pandas as pd

df = pd.read_csv("train.csv")

In [3]:

# Basic metadata check
print(df.columns)
print("----------------------------------------------------------------------------")
print(df.shape)
print("----------------------------------------------------------------------------")
print(df.head())
print("----------------------------------------------------------------------------")
print(df.info())
print("----------------------------------------------------------------------------")
print(df.isnull().sum())

Index(['eeg_id', 'eeg_sub_id', 'eeg_label_offset_seconds', 'spectrogram_id',
       'spectrogram_sub_id', 'spectrogram_label_offset_seconds', 'label_id',
       'patient_id', 'expert_consensus', 'seizure_vote', 'lpd_vote',
       'gpd_vote', 'lrda_vote', 'grda_vote', 'other_vote'],
      dtype='object')
----------------------------------------------------------------------------
(106800, 15)
----------------------------------------------------------------------------
       eeg_id  eeg_sub_id  eeg_label_offset_seconds  spectrogram_id  \
0  1628180742           0                       0.0          353733   
1  1628180742           1                       6.0          353733   
2  1628180742           2                       8.0          353733   
3  1628180742           3                      18.0          353733   
4  1628180742           4                      24.0          353733   

   spectrogram_sub_id  spectrogram_label_offset_seconds    label_id  \
0                   0         

## What we know from the metadata

- The dataset has **106,800 rows** and **15 columns**.
- There are **no missing values in the metadata**.
- `expert_consensus` gives us a clear first target for multiclass modeling.
- The vote columns are clean integer counts.

## Next step in the workflow

1. Understand class balance  
2. Check patient and EEG repetition  
3. Measure label agreement  
4. Split by patient to reduce leakage

In [4]:

# Check target class distribution
print(df["expert_consensus"].value_counts())
print()
print(df["expert_consensus"].value_counts(normalize=True) * 100)

expert_consensus
Seizure    20933
GRDA       18861
Other      18808
GPD        16702
LRDA       16640
LPD        14856
Name: count, dtype: int64

expert_consensus
Seizure    19.600187
GRDA       17.660112
Other      17.610487
GPD        15.638577
LRDA       15.580524
LPD        13.910112
Name: proportion, dtype: float64


In [5]:

# Check how many unique patients, EEGs, and labels we have
print("Unique patients:", df["patient_id"].nunique())
print("Unique eeg_id:", df["eeg_id"].nunique())
print("Unique label_id:", df["label_id"].nunique())
print()

print(df.groupby("patient_id").size().describe())

Unique patients: 1950
Unique eeg_id: 17089
Unique label_id: 106800

count    1950.000000
mean       54.769231
std       133.152047
min         1.000000
25%         8.000000
50%        19.000000
75%        47.000000
max      2215.000000
dtype: float64


In [6]:

# Check vote strength to see how much expert agreement we have
vote_cols = ["seizure_vote", "lpd_vote", "gpd_vote", "lrda_vote", "grda_vote", "other_vote"]

df["total_votes"] = df[vote_cols].sum(axis=1)
df["max_votes"] = df[vote_cols].max(axis=1)
df["agreement_ratio"] = df["max_votes"] / df["total_votes"]

print(df["total_votes"].describe())
print()
print(df["agreement_ratio"].describe())

count    106800.000000
mean          7.255496
std           5.645681
min           1.000000
25%           3.000000
50%           3.000000
75%          13.000000
max          28.000000
Name: total_votes, dtype: float64

count    106800.000000
mean          0.816009
std           0.208296
min           0.200000
25%           0.666667
50%           0.928571
75%           1.000000
max           1.000000
Name: agreement_ratio, dtype: float64


## What these early results mean

### 1. Class imbalance exists, but it is not severe
- The classes are not perfectly balanced, but they are not wildly extreme either.
- **Seizure** is the largest class at about **19.6%**.
- **LPD** is the smallest class at about **13.9%**.

### 2. Patient repetition matters a lot
- We have **106,800 rows**, but only **1,950 unique patients**.
- On average, each patient appears about **54.8 times**.
- One patient appears as many as **2,215 times**.

This means a random row split would be risky because the model could see very similar patterns from the same patient in both training and validation.

### 3. Label agreement is mostly high, but not perfect
- Mean agreement ratio ≈ **0.816**
- Median agreement ratio ≈ **0.929**
- Minimum = **0.20**
- Maximum = **1.00**

Many EEG segments have strong expert agreement, but some are clearly ambiguous. That matches the real-world difficulty of EEG classification.

## Step 2: Open one EEG parquet file and extract the labeled 10-second segment

In [7]:

eeg_folder = "train_eegs"
sample_row = df.iloc[0]

print(sample_row[["eeg_id", "eeg_label_offset_seconds", "expert_consensus"]])

eeg_id                      1628180742
eeg_label_offset_seconds           0.0
expert_consensus               Seizure
Name: 0, dtype: object


In [8]:

# Load one EEG parquet file
import os
import pandas as pd

eeg_id = sample_row["eeg_id"]
eeg_path = os.path.join(eeg_folder, f"{eeg_id}.parquet")

print("Trying to open:", eeg_path)

eeg_data = pd.read_parquet(eeg_path)

print("EEG shape:", eeg_data.shape)
print()
print(eeg_data.head())
print()
print("Columns:")
print(eeg_data.columns.tolist())

Trying to open: train_eegs\1628180742.parquet
EEG shape: (18000, 20)

         Fp1         F3         C3          P3          F7          T3  \
0 -80.519997 -70.540001 -80.110001 -108.750000 -120.330002  -88.620003   
1 -80.449997 -70.330002 -81.760002 -107.669998 -120.769997  -90.820000   
2 -80.209999 -75.870003 -82.050003 -106.010002 -117.500000  -87.489998   
3 -84.709999 -75.339996 -87.480003 -108.970001 -121.410004  -94.750000   
4 -90.570000 -80.790001 -93.000000 -113.870003 -129.960007 -102.860001   

           T5          O1          Fz          Cz         Pz        Fp2  \
0 -101.750000 -104.489998  -99.129997  -90.389999 -97.040001 -77.989998   
1 -104.260002  -99.730003  -99.070000  -92.290001 -96.019997 -84.500000   
2  -99.589996  -96.820000 -119.680000  -99.360001 -91.110001 -99.440002   
3 -105.370003 -100.279999 -113.839996 -102.059998 -95.040001 -99.230003   
4 -118.599998 -101.099998 -107.660004 -102.339996 -98.510002 -95.300003   

           F4          C4         

In [9]:

# Extract the labeled 10-second segment
FS = 200

offset_seconds = sample_row["eeg_label_offset_seconds"]

start = int(offset_seconds * FS + 20 * FS)
end = start + 10 * FS

segment = eeg_data.iloc[start:end].copy()

print("Start row:", start)
print("End row:", end)
print("Segment shape:", segment.shape)
print()
print(segment.head())

Start row: 4000
End row: 6000
Segment shape: (2000, 20)

             Fp1          F3          C3          P3          F7          T3  \
4000 -116.839996 -123.959999 -128.309998 -164.490005 -139.279999 -113.559998   
4001 -118.199997 -130.479996 -132.850006 -168.889999 -134.990005 -127.180000   
4002 -118.269997 -128.360001 -128.410004 -166.669998 -137.570007 -119.190002   
4003 -110.550003 -118.169998 -122.550003 -164.160004 -136.470001 -114.480003   
4004 -110.629997 -123.209999 -126.940002 -166.259995 -135.369995 -129.320007   

              T5          O1          Fz          Cz          Pz         Fp2  \
4000 -128.679993 -140.050003 -134.000000 -118.510002 -127.300003 -119.559998   
4001 -132.789993 -141.539993 -144.759995 -120.440002 -126.029999 -123.849998   
4002 -132.289993 -143.559998 -143.199997 -117.580002 -121.800003 -124.500000   
4003 -137.429993 -144.970001 -124.660004 -106.459999 -118.870003 -104.239998   
4004 -140.660004 -146.580002 -130.179993 -107.379997 -117.4400

In [10]:

# Check missing values in the extracted segment
print(segment.isnull().sum())

Fp1    0
F3     0
C3     0
P3     0
F7     0
T3     0
T5     0
O1     0
Fz     0
Cz     0
Pz     0
Fp2    0
F4     0
C4     0
P4     0
F8     0
T4     0
T6     0
O2     0
EKG    0
dtype: int64


In [11]:

# Extract a richer set of signal features from one segment
def extract_features(segment):
    feature_row = {}

    for col in segment.columns:
        if col == "EKG":
            continue

        s = segment[col].copy()
        s = s.fillna(s.median())

        q1 = s.quantile(0.25)
        q3 = s.quantile(0.75)

        feature_row[col + "_mean"] = s.mean()
        feature_row[col + "_std"] = s.std()
        feature_row[col + "_min"] = s.min()
        feature_row[col + "_max"] = s.max()
        feature_row[col + "_median"] = s.median()
        feature_row[col + "_iqr"] = q3 - q1
        feature_row[col + "_range"] = s.max() - s.min()

        feature_row[col + "_abs_mean"] = s.abs().mean()
        feature_row[col + "_rms"] = (s.pow(2).mean()) ** 0.5
        feature_row[col + "_energy"] = s.pow(2).sum()
        feature_row[col + "_skew"] = s.skew()
        feature_row[col + "_kurt"] = s.kurt()

        zero_crossings = ((s.shift(1) * s) < 0).sum()
        feature_row[col + "_zero_cross"] = zero_crossings

    return feature_row

one_feature_row = extract_features(segment)

print("Number of features:", len(one_feature_row))
print(list(one_feature_row.items())[:10])

Number of features: 247
[('Fp1_mean', np.float32(-123.16487)), ('Fp1_std', np.float32(27.188683)), ('Fp1_min', np.float32(-208.05)), ('Fp1_max', np.float32(-37.96)), ('Fp1_median', np.float32(-122.915)), ('Fp1_iqr', np.float64(35.28750038146973)), ('Fp1_range', np.float32(170.09)), ('Fp1_abs_mean', np.float32(123.16487)), ('Fp1_rms', np.float32(126.12867)), ('Fp1_energy', np.float32(3.181688e+07))]


## Step 3: Split the metadata by `patient_id`
We split by patient instead of by row so that the same patient's EEG patterns do not leak into both training and validation.

In [12]:

from sklearn.model_selection import GroupShuffleSplit

X_meta = df.drop(columns=["expert_consensus"])
y = df["expert_consensus"]
groups = df["patient_id"]

splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, valid_idx = next(splitter.split(X_meta, y, groups))

meta_train = df.iloc[train_idx].reset_index(drop=True)
meta_valid = df.iloc[valid_idx].reset_index(drop=True)

print("Training rows:", meta_train.shape[0])
print("Validation rows:", meta_valid.shape[0])
print("Training patients:", meta_train["patient_id"].nunique())
print("Validation patients:", meta_valid["patient_id"].nunique())

train_patients = set(meta_train["patient_id"])
valid_patients = set(meta_valid["patient_id"])

print("Patient overlap:", len(train_patients.intersection(valid_patients)))

Training rows: 83430
Validation rows: 23370
Training patients: 1560
Validation patients: 390
Patient overlap: 0


In [13]:

# Reusable helper functions for segment loading and feature extraction
import os
import pandas as pd

FS = 200

def load_labeled_segment(row, eeg_folder):
    eeg_id = row["eeg_id"]
    eeg_path = os.path.join(eeg_folder, f"{eeg_id}.parquet")

    eeg_data = pd.read_parquet(eeg_path)

    offset_seconds = row["eeg_label_offset_seconds"]
    start = int(offset_seconds * FS + 20 * FS)
    end = start + 10 * FS

    segment = eeg_data.iloc[start:end].copy()
    return segment


def extract_features(segment):
    feature_row = {}

    for col in segment.columns:
        if col == "EKG":
            continue

        s = segment[col].copy()
        s = s.fillna(s.median())

        q1 = s.quantile(0.25)
        q3 = s.quantile(0.75)

        feature_row[col + "_mean"] = s.mean()
        feature_row[col + "_std"] = s.std()
        feature_row[col + "_min"] = s.min()
        feature_row[col + "_max"] = s.max()
        feature_row[col + "_median"] = s.median()
        feature_row[col + "_iqr"] = q3 - q1
        feature_row[col + "_range"] = s.max() - s.min()

        feature_row[col + "_abs_mean"] = s.abs().mean()
        feature_row[col + "_rms"] = (s.pow(2).mean()) ** 0.5
        feature_row[col + "_energy"] = s.pow(2).sum()
        feature_row[col + "_skew"] = s.skew()
        feature_row[col + "_kurt"] = s.kurt()

        zero_crossings = ((s.shift(1) * s) < 0).sum()
        feature_row[col + "_zero_cross"] = zero_crossings

    return feature_row

In [14]:

# For this pilot experiment, we are not using the full dataset yet.
# We sample 8,000 training rows and 2,000 validation rows so we can iterate faster.

eeg_folder = "train_eegs"

pilot_train_meta = meta_train.sample(n=8000, random_state=42)
pilot_valid_meta = meta_valid.sample(n=2000, random_state=42)

def build_feature_table(meta_df, eeg_folder):
    rows = []

    for i in range(len(meta_df)):
        row = meta_df.iloc[i]

        try:
            segment = load_labeled_segment(row, eeg_folder)
            features = extract_features(segment)

            features["target"] = row["expert_consensus"]
            features["patient_id"] = row["patient_id"]
            features["eeg_id"] = row["eeg_id"]
            features["label_id"] = row["label_id"]

            rows.append(features)

        except Exception as e:
            print(f"Skipped row {i} because of error: {e}")

    return pd.DataFrame(rows)

train_features = build_feature_table(pilot_train_meta, eeg_folder)
valid_features = build_feature_table(pilot_valid_meta, eeg_folder)

print("Train feature shape:", train_features.shape)
print("Valid feature shape:", valid_features.shape)

print(train_features["target"].value_counts())
print()
print(valid_features["target"].value_counts())

Train feature shape: (8000, 251)
Valid feature shape: (2000, 251)
target
Seizure    1540
Other      1491
GRDA       1482
LRDA       1288
LPD        1117
GPD        1082
Name: count, dtype: int64

target
GPD        417
Seizure    373
GRDA       336
Other      325
LPD        286
LRDA       263
Name: count, dtype: int64


## Step 4: Train baseline and comparison models
We start with a simple baseline, then compare it with tree-based models that usually perform better on structured tabular features.

In [15]:

from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

feature_cols = [col for col in train_features.columns if col not in ["target", "patient_id", "eeg_id", "label_id"]]

X_train = train_features[feature_cols]
X_valid = valid_features[feature_cols]

le = LabelEncoder()
y_train = le.fit_transform(train_features["target"])
y_valid = le.transform(valid_features["target"])

lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train, y_train)

y_pred = lr_model.predict(X_valid)

print("Classes:", le.classes_)
print()
print(classification_report(y_valid, y_pred, target_names=le.classes_))
print()
print(confusion_matrix(y_valid, y_pred))

Classes: ['GPD' 'GRDA' 'LPD' 'LRDA' 'Other' 'Seizure']

              precision    recall  f1-score   support

         GPD       0.29      0.02      0.04       417
        GRDA       0.20      0.00      0.01       336
         LPD       0.19      0.11      0.14       286
        LRDA       0.02      0.00      0.01       263
       Other       0.18      0.38      0.25       325
     Seizure       0.17      0.50      0.26       373

    accuracy                           0.18      2000
   macro avg       0.18      0.17      0.12      2000
weighted avg       0.19      0.18      0.12      2000


[[  9   0  15  30  43 320]
 [  3   1  21   6 175 130]
 [  1   0  31   2 102 150]
 [  0   0  26   1 100 136]
 [  8   4  29   1 123 160]
 [ 10   0  44   1 132 186]]


C:\Users\Renuka\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [16]:

from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
import pandas as pd

results = []

def evaluate_model(model_name, y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average="macro")
    weighted_f1 = f1_score(y_true, y_pred, average="weighted")

    results.append({
        "Model": model_name,
        "Accuracy": acc,
        "Macro F1": macro_f1,
        "Weighted F1": weighted_f1
    })

    print(f"\n{model_name}")
    print("Accuracy:", acc)
    print("Macro F1:", macro_f1)
    print("Weighted F1:", weighted_f1)
    print()
    print(classification_report(y_true, y_pred, target_names=le.classes_))
    print()
    print(confusion_matrix(y_true, y_pred))

In [17]:

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

feature_cols = [col for col in train_features.columns
                if col not in ["target", "patient_id", "eeg_id", "label_id"]]

X_train = train_features[feature_cols]
X_valid = valid_features[feature_cols]

from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()

y_train = le.fit_transform(train_features["target"])
y_valid = le.transform(valid_features["target"])

lr_scaled = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=2000, class_weight="balanced"))
])

lr_scaled.fit(X_train, y_train)
lr_scaled_pred = lr_scaled.predict(X_valid)

evaluate_model("Logistic Regression (scaled)", y_valid, lr_scaled_pred)


Logistic Regression (scaled)
Accuracy: 0.3135
Macro F1: 0.2927376982549263
Weighted F1: 0.3093559745655701

              precision    recall  f1-score   support

         GPD       0.49      0.56      0.52       417
        GRDA       0.29      0.24      0.26       336
         LPD       0.20      0.14      0.17       286
        LRDA       0.18      0.26      0.21       263
       Other       0.25      0.23      0.24       325
     Seizure       0.36      0.34      0.35       373

    accuracy                           0.31      2000
   macro avg       0.29      0.30      0.29      2000
weighted avg       0.31      0.31      0.31      2000


[[233  19  38  29  41  57]
 [ 26  82  25  90  83  30]
 [ 77  28  41  59  24  57]
 [ 38  57  31  68  30  39]
 [ 45  53  33  77  76  41]
 [ 60  47  35  54  50 127]]


In [18]:

from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight="balanced"
)

rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_valid)

evaluate_model("Random Forest", y_valid, rf_pred)


Random Forest
Accuracy: 0.3035
Macro F1: 0.28089695760308975
Weighted F1: 0.2931763708826844

              precision    recall  f1-score   support

         GPD       0.55      0.27      0.36       417
        GRDA       0.23      0.31      0.26       336
         LPD       0.34      0.10      0.15       286
        LRDA       0.19      0.16      0.17       263
       Other       0.30      0.41      0.35       325
     Seizure       0.32      0.50      0.39       373

    accuracy                           0.30      2000
   macro avg       0.32      0.29      0.28      2000
weighted avg       0.33      0.30      0.29      2000


[[114  95  17  14  23 154]
 [  8 105   3  70  66  84]
 [ 37  68  28  30  75  48]
 [ 20  68  16  41  70  48]
 [ 11  69   5  42 134  64]
 [ 18  59  14  24  73 185]]


In [19]:

results_df = pd.DataFrame(results).drop_duplicates().sort_values("Accuracy", ascending=False).reset_index(drop=True)
print(results_df)

                          Model  Accuracy  Macro F1  Weighted F1
0  Logistic Regression (scaled)    0.3135  0.292738     0.309356
1                 Random Forest    0.3035  0.280897     0.293176


In [20]:

# Install XGBoost only if it is missing
import importlib.util
import sys

if importlib.util.find_spec("xgboost") is None:
    !{sys.executable} -m pip install xgboost

In [21]:

from xgboost import XGBClassifier
from sklearn.utils.class_weight import compute_sample_weight

sample_weights = compute_sample_weight(class_weight="balanced", y=y_train)

xgb_model = XGBClassifier(
    objective="multi:softmax",
    num_class=len(le.classes_),
    n_estimators=500,
    max_depth=4,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric="mlogloss"
)

xgb_model.fit(X_train, y_train, sample_weight=sample_weights)

xgb_pred = xgb_model.predict(X_valid)

evaluate_model("XGBoost", y_valid, xgb_pred)


XGBoost
Accuracy: 0.397
Macro F1: 0.37439408490939075
Weighted F1: 0.39218981881941056

              precision    recall  f1-score   support

         GPD       0.60      0.57      0.59       417
        GRDA       0.30      0.32      0.31       336
         LPD       0.31      0.19      0.23       286
        LRDA       0.28      0.25      0.26       263
       Other       0.35      0.43      0.39       325
     Seizure       0.43      0.50      0.46       373

    accuracy                           0.40      2000
   macro avg       0.38      0.38      0.37      2000
weighted avg       0.39      0.40      0.39      2000


[[239  52  38   9  17  62]
 [ 21 108  20  63  54  70]
 [ 68  53  54  21  67  23]
 [ 21  50  27  65  59  41]
 [ 16  65  11  41 141  51]
 [ 31  36  25  33  61 187]]


In [22]:

results_df = pd.DataFrame(results).drop_duplicates().sort_values("Accuracy", ascending=False).reset_index(drop=True)
print(results_df)

# We first tested much smaller pilot samples, then increased them to 8,000 training rows
# and 2,000 validation rows. The larger pilot set gave us more stable and more reliable
# model comparisons.

                          Model  Accuracy  Macro F1  Weighted F1
0                       XGBoost    0.3970  0.374394     0.392190
1  Logistic Regression (scaled)    0.3135  0.292738     0.309356
2                 Random Forest    0.3035  0.280897     0.293176


## Model summary so far

Among the models we tested, **XGBoost** performed the best on the pilot validation set.  
Our updated results show that:

- stronger signal features helped all models,
- XGBoost handled the nonlinear tabular feature patterns best,
- model choice and feature quality both had a clear effect on performance.

At this point, XGBoost is our strongest candidate for the final version of the project.

In [23]:

from sklearn.ensemble import HistGradientBoostingClassifier

hgb_model = HistGradientBoostingClassifier(
    max_iter=300,
    learning_rate=0.05,
    max_depth=6,
    random_state=42
)

hgb_model.fit(X_train, y_train)
hgb_pred = hgb_model.predict(X_valid)

evaluate_model("HistGradientBoosting", y_valid, hgb_pred)


HistGradientBoosting
Accuracy: 0.3615
Macro F1: 0.3386615801676766
Weighted F1: 0.3570435704888242

              precision    recall  f1-score   support

         GPD       0.74      0.41      0.53       417
        GRDA       0.26      0.36      0.30       336
         LPD       0.33      0.15      0.20       286
        LRDA       0.26      0.15      0.19       263
       Other       0.29      0.46      0.35       325
     Seizure       0.39      0.54      0.45       373

    accuracy                           0.36      2000
   macro avg       0.38      0.34      0.34      2000
weighted avg       0.40      0.36      0.36      2000


[[172  88  33   5  37  82]
 [  9 120   6  46  78  77]
 [ 16  81  42  16  75  56]
 [ 16  58  16  40  83  50]
 [ 11  72  13  25 148  56]
 [  9  35  18  19  91 201]]
